# Exact Significance Test for Term Dispersion Toy Example

Author: Paul Sheridan

Description: A toy example demonstrating a correspondence between the RICF scores of Eq. (3) and the term dispersion p-values of Eq. (4). Results reported in Table 4.

## Initial Setup

In [1]:
# Imports
import numpy as np
import pandas as pd
import sys
sys.path.append('../')
import wordstats as ws

# Initialize a random seed to ensure results can be replicated
rng = np.random.default_rng(998510)

## Toy Document Collection

Hardcode a toy document collection.

In [2]:
d = 8 # Number of docs.
m = 6 # Vocab size.

# Define term-in-doc matrix. Terms are rows, docs are columns.
nij = np.zeros((m, d), dtype=int)
nij[0, :] = [3, 0, 2, 0, 2, 0, 0, 0]
nij[1, :] = [0, 0, 4, 0, 0, 0, 0, 0]
nij[2, :] = [5, 3, 1, 4, 0, 6, 0, 5]
nij[3, :] = [2, 4, 3, 2, 2, 3, 4, 0]
nij[4, :] = [0, 0, 0, 0, 3, 0, 2, 0]
nij[5, :] = [0, 3, 0, 4, 3, 1, 4, 5]

# Create associated Pandas data frame for purposes of printing to console.
term_names = [f"t{i}" for i in range(1, m + 1)]
doc_names = [f"d{j}" for j in range(1, d + 1)]
nij_df = pd.DataFrame(nij, index=term_names, columns=doc_names)

# Print term-in-doc matrix to console.
print("Term-in-doc matrix:\n")
display(nij_df)

# Calculate word statistics.
ri = nij.sum(axis=1) # Collection level term frequencies.
nj = nij.sum(axis=0) # Doc lengths.
bij = (nij > 0).astype(int) # Binary presence/absence.
bi = bij.sum(axis=1) # Document frequencies per term.
n = int(nj.sum()) # Total terms in collection.
theta_i = ri / n # Collection level term proportions.

# Print term statistics to console.
print("\nCollection level term frequencies (ri):\n")
display(pd.Series(ri, index=term_names, name="ri"))

print("\nCollection level term proportions (theta_i):\n")
display(pd.Series(theta_i, index=term_names, name="theta_i"))

print("\nDoc lengths (nj):\n")
display(pd.Series(nj, index=doc_names, name="nj"))

print("\nDocument frequencies (bi):\n")
display(pd.Series(bi, index=term_names, name="bi"))

print(f"\nTotal terms n = {n}\n")

Term-in-doc matrix:



,d1,d2,d3,d4,d5,d6,d7,d8
t1,3,0,2,0,2,0,0,0
t2,0,0,4,0,0,0,0,0
t3,5,3,1,4,0,6,0,5
t4,2,4,3,2,2,3,4,0
t5,0,0,0,0,3,0,2,0
t6,0,3,0,4,3,1,4,5



Collection level term frequencies (ri):



t1     7
t2     4
t3    24
t4    20
t5     5
t6    20
Name: ri, dtype: int64


Collection level term proportions (theta_i):



...


Doc lengths (nj):



d1    10
d2    10
d3    10
d4    10
d5    10
d6    10
d7    10
d8    10
Name: nj, dtype: int64


Document frequencies (bi):



t1    3
t2    1
t3    6
t4    7
t5    2
t6    6
Name: bi, dtype: int64


Total terms n = 80



## RICF Scores

Calculate the RICF scores of Eq. (3).

In [3]:
# Convert word these word statistics to format accepted by processing functions.
ri_mat = np.matrix(ri.reshape(1, m))
nj_mat = np.matrix(nj.reshape(d, 1))
bi_mat = np.matrix(bi.reshape(1, m))

# Calculate observed ICF scores.
CF = ws.get_CF(ri_mat)
ICF = ws.get_ICF(CF)

# Estimate the theta values used to calculate expected ICF values.
thetas = np.linspace(0.001, 0.5, num=1000)
opt_thetas = ws.get_opt_thetas(n, m, d, ri_mat, nj_mat, bi_mat, thetas)

# Calculate RICF scores.
RICF = np.asarray(ws.get_RICF(opt_thetas, n, ICF)).flatten()

# Print RIFC scores to console.
print("\nRICF scores:\n")
display(pd.Series(RICF, index=term_names, name="RICF"))


RICF scores:



t1    0.646561
t2    1.331196
t3    0.841173
t4    0.286847
t5    0.792705
t6    0.658852
Name: RICF, dtype: float64

## Estimate Significance Test p-values

Estimate significance test p-values of Eq. (4) using a simple Monte Carlo sampling scheme.

In [4]:
# Initialize settings.
R_sat = 100_000 # Set number of required successful simulations (per term).
pvalues_est = np.zeros(m, dtype=float) # Initialize estimated p-values (per term).

# Estimate p-value for each term.
for i in range(m):
    ri_obs = int(ri[i]) # Observed collection level term frequency for term i.
    R_numer = 0 # Counter for it simulated collection satisfies ri > ri_obs and basic conditions.
    R_denom = 0 # Counter for if simulated collection fulfills basic conditions.

    # Simulate many document collections and keep track of which fulfill conditions.
    while R_denom < R_sat:
        nij_sim = np.zeros((m, d), dtype=int)

        # Simulate each document column as multinomial.
        for j in range(d):
            nij_sim[:, j] = rng.multinomial(int(nj[j] - 1), theta_i)

        nj_sim = nij_sim.sum(axis=0)

        # If empty doc exists, discard.
        if not np.any(nj_sim == 0):
            row = nij_sim[i, :]
            ri_sim = int(row.sum())
            bi_sim = int((row > 0).sum())

            if bi[i] == bi_sim:
                R_denom += 1

            if (ri_sim >= ri_obs) and (bi[i] == bi_sim):
                R_numer += 1

    # Estimated p-value is the number of successful cases divided by the number of total cases.
    pvalues_est[i] = R_numer / R_denom

    # Print result to console.
    print(f"\nTerm: {i+1}")
    print(f"Numerator: {R_numer}")
    print(f"Denominator: {R_denom}")
    print(f"Estimated p-value: {pvalues_est[i]}")
    print(f"Negative log10 probability: {-np.log10(pvalues_est[i])}")



Term: 1
Numerator: 3758
Denominator: 100000
Estimated p-value: 0.03758
Negative log10 probability: 1.4250432242354931

Term: 2
Numerator: 179
Denominator: 100000
Estimated p-value: 0.00179
Negative log10 probability: 2.7471469690201067

Term: 3
Numerator: 2187
Denominator: 100000
Estimated p-value: 0.02187
Negative log10 probability: 1.660151216962363

Term: 4
Numerator: 20819
Denominator: 100000
Estimated p-value: 0.20819
Negative log10 probability: 0.6815401348116489

Term: 5
Numerator: 1946
Denominator: 100000
Estimated p-value: 0.01946
Negative log10 probability: 1.7108571640676669

Term: 6
Numerator: 5023
Denominator: 100000
Estimated p-value: 0.05023
Negative log10 probability: 1.2990368218404507
